# Order-to-Cash Process Analysis
**End-to-end operational analysis using synthetic order, fulfilment, invoice and payment data.**

This notebook creates a reproducible sample dataset, measures process performance, identifies bottlenecks and translates findings into practical improvement opportunities. All data is synthetic and generated in the notebook.

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas numpy matplotlib seaborn jupyter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Generate sample order-to-cash data

In [ ]:
n = 1200
start = pd.Timestamp("2025-01-01")
order_date = start + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
order_date = pd.Series(order_date).sort_values().reset_index(drop=True)

channels = rng.choice(["Web", "Sales Team", "Marketplace"], n, p=[0.55, 0.30, 0.15])
regions = rng.choice(["West", "North", "South", "East"], n)
products = rng.choice(["Hardware", "Software", "Accessories", "Services"], n, p=[.35,.25,.25,.15])
order_value = np.round(rng.lognormal(mean=8.2, sigma=.65, size=n), 2)

# Delays vary by channel and region to create realistic operational patterns.
channel_delay = pd.Series(channels).map({"Web":0, "Sales Team":1.0, "Marketplace":1.8}).to_numpy()
region_delay = pd.Series(regions).map({"West":0, "North":.5, "South":1.1, "East":1.6}).to_numpy()
processing_days = np.maximum(.2, rng.gamma(1.6, 1.1, n) + channel_delay*.25)
fulfilment_days = np.maximum(.5, rng.gamma(2.1, 2.0, n) + region_delay + (products=="Hardware")*1.2)
invoice_days = np.maximum(.1, rng.gamma(1.2, .8, n) + (channels=="Sales Team")*.4)
payment_days = np.maximum(1, rng.gamma(2.5, 7.0, n) + (channels=="Marketplace")*4)

order_approved = order_date + pd.to_timedelta(processing_days, unit="D")
fulfilled = order_approved + pd.to_timedelta(fulfilment_days, unit="D")
invoiced = fulfilled + pd.to_timedelta(invoice_days, unit="D")
paid = invoiced + pd.to_timedelta(payment_days, unit="D")

# Some orders remain open/unpaid at the reporting cut-off.
cutoff = pd.Timestamp("2026-01-15")
paid_flag = paid <= cutoff
payment_date = pd.Series(paid).where(paid_flag, pd.NaT)

df = pd.DataFrame({
    "order_id":[f"ORD-{i:05d}" for i in range(1,n+1)],
    "order_date":order_date,
    "channel":channels, "region":regions, "product_category":products,
    "order_value":order_value,
    "approval_date":order_approved,
    "fulfilment_date":fulfilled,
    "invoice_date":invoiced,
    "payment_date":payment_date
})
df["order_cycle_days"] = (df["fulfilment_date"]-df["order_date"]).dt.total_seconds()/86400
df["invoice_cycle_days"] = (df["invoice_date"]-df["fulfilment_date"]).dt.total_seconds()/86400
df["days_to_payment"] = (df["payment_date"]-df["invoice_date"]).dt.total_seconds()/86400
df["order_to_cash_days"] = (df["payment_date"]-df["order_date"]).dt.total_seconds()/86400
df["payment_status"] = np.where(df["payment_date"].notna(), "Paid", "Outstanding")
df["on_time_fulfilment"] = df["order_cycle_days"] <= 7

df.head()


## 2. Data quality and overall KPIs

In [ ]:
print("Rows:", len(df))
print("Duplicate order IDs:", df["order_id"].duplicated().sum())
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

kpis = pd.Series({
    "Total orders": len(df),
    "Total order value": df["order_value"].sum(),
    "Average order-to-fulfilment (days)": df["order_cycle_days"].mean(),
    "Median order-to-fulfilment (days)": df["order_cycle_days"].median(),
    "On-time fulfilment rate (%)": df["on_time_fulfilment"].mean()*100,
    "Paid orders (%)": df["payment_status"].eq("Paid").mean()*100,
    "Average days to payment (paid orders)": df["days_to_payment"].mean(),
    "Outstanding order value": df.loc[df["payment_status"].eq("Outstanding"), "order_value"].sum()
})
display(kpis.to_frame("value"))


## 3. Process-stage analysis

In [ ]:
stage_metrics = pd.Series({
    "Order approval": (df["approval_date"]-df["order_date"]).dt.total_seconds().mean()/86400,
    "Fulfilment": (df["fulfilment_date"]-df["approval_date"]).dt.total_seconds().mean()/86400,
    "Invoicing": df["invoice_cycle_days"].mean(),
    "Payment (paid only)": df["days_to_payment"].mean()
}).sort_values(ascending=False)

display(stage_metrics.to_frame("average_days"))
ax = stage_metrics.sort_values().plot(kind="barh", figsize=(9,4), title="Average time by process stage")
ax.set_xlabel("Average days")
plt.tight_layout()
plt.show()


## 4. Identify delays by channel, region and product

In [ ]:
by_channel = df.groupby("channel").agg(
    orders=("order_id","count"),
    avg_fulfilment_days=("order_cycle_days","mean"),
    on_time_rate=("on_time_fulfilment","mean"),
    outstanding_value=("order_value", lambda s: s[df.loc[s.index,"payment_status"].eq("Outstanding")].sum())
).sort_values("avg_fulfilment_days", ascending=False)
by_channel["on_time_rate"] *= 100
display(by_channel.round(2))

by_region = df.groupby("region").agg(
    orders=("order_id","count"),
    avg_fulfilment_days=("order_cycle_days","mean"),
    on_time_rate=("on_time_fulfilment","mean")
)
by_region["on_time_rate"] *= 100
display(by_region.round(2))

plt.figure(figsize=(9,4))
sns.boxplot(data=df, x="region", y="order_cycle_days")
plt.axhline(7, color="red", linestyle="--", label="7-day target")
plt.title("Order-to-fulfilment time by region")
plt.ylabel("Days")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Monthly performance trend

In [ ]:
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
monthly = df.groupby("order_month").agg(
    orders=("order_id","count"),
    avg_cycle_days=("order_cycle_days","mean"),
    on_time_rate=("on_time_fulfilment","mean"),
    order_value=("order_value","sum")
)
monthly["on_time_rate"] *= 100
display(monthly.head())

fig, ax1 = plt.subplots(figsize=(12,4))
monthly["avg_cycle_days"].plot(ax=ax1, marker="o", label="Average cycle days")
ax1.set_ylabel("Average days")
ax2 = ax1.twinx()
monthly["on_time_rate"].plot(ax=ax2, color="green", marker="s", label="On-time rate (%)")
ax2.set_ylabel("On-time fulfilment (%)")
ax1.set_title("Monthly order fulfilment performance")
fig.tight_layout()
plt.show()


## 6. Outstanding payments and improvement opportunities

In [ ]:
outstanding = df[df["payment_status"].eq("Outstanding")].copy()
outstanding["days_since_invoice"] = (cutoff - outstanding["invoice_date"]).dt.days
outstanding["aging_bucket"] = pd.cut(
    outstanding["days_since_invoice"],
    bins=[-1, 7, 15, 30, 60, np.inf],
    labels=["0-7 days","8-15 days","16-30 days","31-60 days","60+ days"]
)
aging = outstanding.groupby("aging_bucket", observed=False).agg(
    invoices=("order_id","count"), outstanding_value=("order_value","sum")
)
display(aging)

# Simple, transparent rule-based recommendations based on observed bottlenecks.
bottleneck = stage_metrics.idxmax()
slow_channel = by_channel["avg_fulfilment_days"].idxmax()
late_region = by_region["avg_fulfilment_days"].idxmax()
print("Data-led improvement opportunities")
print(f"1. Prioritise the {bottleneck.lower()} stage: it has the longest average duration.")
print(f"2. Review hand-offs and capacity in the {slow_channel} channel, which has the slowest fulfilment cycle.")
print(f"3. Investigate inventory availability, carrier performance and regional dispatch for {late_region}.")
print("4. Automate invoice creation after proof of delivery and introduce exception alerts for overdue invoices.")
print("5. Segment collections by invoice ageing and send reminders before due dates.")


## 7. Export analysis outputs

In [ ]:
df.to_csv(OUTPUT_DIR / "order_to_cash_sample_data.csv", index=False)
monthly.to_csv(OUTPUT_DIR / "monthly_order_to_cash_kpis.csv")
by_channel.to_csv(OUTPUT_DIR / "channel_performance.csv")
aging.to_csv(OUTPUT_DIR / "outstanding_invoice_aging.csv")
print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")


## 8. Conclusion
Use the KPI tables and charts to explain where time is spent across the order-to-cash lifecycle. In a real business setting, validate the target thresholds with stakeholders, replace synthetic data with approved operational extracts, and confirm whether delays are caused by inventory, approvals, fulfilment capacity, invoicing controls or customer payment behaviour.

**Portfolio note:** This is a synthetic-data portfolio project. Do not present generated values as actual company performance.